# Exploratory Data Analysis of AdoptAI System Metrics

This dataset contains system-performance measurements collected from several machines. The data has already been merged and cleaned.

The objective is to understand its structure, quality, distributions, outliers, correlations, reliability, machine differences, and changes over time before preprocessing and machine learning.

This notebook only reads and analyzes the data. It does not modify the original CSV file.

## 1. Import Libraries

Import the simple tools needed for data analysis and charts.

In [ ]:
import pandas as pd              # Work with tables
import numpy as np               # Work with numerical values
import matplotlib.pyplot as plt # Create charts
from pathlib import Path         # Build file paths

## 2. Define the Dataset Path

Find the project folder safely, whether the notebook starts from the project root or the `notebooks` folder.

In [ ]:
current_folder = Path.cwd()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
data_path = project_root / "data" / "processed" / "cleaned_metrics.csv"
print("Dataset path:", data_path)

## 3. Load the Dataset

Check that the cleaned CSV exists, then load it without changing the file.

In [ ]:
if data_path.exists():
    df = pd.read_csv(data_path)
    print("Dataset loaded successfully.")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
else:
    df = pd.DataFrame()
    print("Dataset not found. Check the path shown above.")

## 4. Preview the Dataset

These previews help verify the content at the beginning, end, and across random positions.

In [ ]:
# First five rows
df.head()

#### Line-by-line explanation

1. `# First five rows` — Comment explaining the next instruction.
2. `df.head()` — Shows the first five rows.

In [ ]:
# Last five rows
df.tail()

#### Line-by-line explanation

1. `# Last five rows` — Comment explaining the next instruction.
2. `df.tail()` — Shows the last five rows.

### Why do we display random rows?

After loading the dataset, we inspect a few **random rows** to better understand the data.

Unlike `head()`, which always shows the first rows, a random sample lets us check whether the data looks consistent throughout the dataset and helps detect unexpected values.

This step **does not modify the dataset**; it is only for visual inspection during EDA.

In [ ]:
# Random rows, only when data is available
if len(df) >= 5:
    display(df.sample(5, random_state=42))
else:
    print("Not enough rows for a five-row random sample.")

### Preview observation

Check whether the rows and column values look consistent with system measurements from different machines.

## 5. Dataset Structure

Inspect the size, names, types, and non-missing counts.

In [ ]:
print("Dataframe shape:", df.shape)

In [ ]:
print("Column names:")
print(df.columns.tolist())

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("data_type"))

In [ ]:
df.info()

### Structure observation

Note which columns are numbers, text, identifiers, or dates. A wrong data type may need correction during preprocessing.

## 6. Convert Timestamp Columns

Timestamps must be datetime values for time analysis. Invalid text becomes `NaT` but is not removed.

### Goal of this cell

Convert timestamp columns to datetime format so Python can perform time-based analysis and safely detect invalid dates.
So we can later:

analyze data over time,
calculate run durations,
sort by timestamp,
create time-series graphs,
identify invalid timestamps

In [ ]:
timestamp_columns = ["timestamp", "ended_at_utc", "started_at_utc"]
for column in timestamp_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(df[column], errors="coerce", utc=True)
        print(column, "invalid or missing values:", df[column].isna().sum())
    else:
        print(column, "is not available; skipped.")

## 7. Duplicate Analysis

Verify duplicate rows. This notebook reports duplicates but does not delete them.

In [ ]:
print("Total duplicate rows:", df.duplicated().sum())

In [ ]:
if {"run_id", "timestamp"}.issubset(df.columns):
    pair_duplicates = df.duplicated(subset=["run_id", "timestamp"]).sum()
    print("Duplicate run_id and timestamp pairs:", pair_duplicates)
else:
    print("run_id or timestamp is missing; pair check skipped.")

### Duplicate observation

Cleaning should have handled duplicates. Any remaining duplicates should be investigated before modeling.

## 8. Missing-Value Analysis

Count missing values and percentages for every column.

In [ ]:
missing_table = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (df.isna().mean().values * 100).round(2)
}).sort_values("missing_percentage", ascending=False)
missing_columns = missing_table[missing_table["missing_count"] > 0]
display(missing_columns)

### Goal of this cell

After identifying missing values in each column, we now want to know which columns are **complete**.

A complete column contains **no missing values**, meaning every row has a value.

These columns usually require less preprocessing and are generally more reliable for analysis.

In [ ]:
complete_columns = missing_table[missing_table["missing_count"] == 0][["column"]]
print("Columns without missing values:")
display(complete_columns)

In [ ]:
if not missing_columns.empty:
    plt.figure(figsize=(10, max(4, len(missing_columns) * 0.3)))
    plt.barh(missing_columns["column"], missing_columns["missing_percentage"])
    plt.title("Missing Values by Column")
    plt.xlabel("Missing values (%)")
    plt.ylabel("Column")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values were found.")

### Missing-value observation

Look for columns with high missing percentages. Decide later whether missing values mean an unavailable sensor, a collection problem, or a value that needs treatment.

## 9. Descriptive Statistics

Summaries show the center, spread, and range of each variable.

### Goal of this cell

This section generates descriptive statistics for all numerical columns in the dataset.

It helps us understand the distribution of each metric by displaying values such as the minimum, maximum, mean, median, and standard deviation.

This analysis allows us to detect unusual values, understand the range of the data, and prepare for future preprocessing and machine learning.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
if numeric_columns:
    display(df[numeric_columns].describe().T)
else:
    print("No numerical columns are available.")

In [ ]:
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
if categorical_columns:
    display(df[categorical_columns].describe(include="all").T)
else:
    print("No categorical columns are available.")

In [ ]:
display(df.nunique(dropna=False).sort_values(ascending=False).to_frame("unique_values"))

### Statistics observation

Compare means and medians. A large difference can suggest a skewed distribution or unusual values.

## 10. Identify Metric Groups

Group available columns into understandable metric families.

In [ ]:
metric_candidates = {
    "CPU": ["cpu_pct", "cpu_frequency_mhz", "cpu_freq_mhz", "load_avg_1m", "load_avg_5m", "load_avg_15m"],
    "Memory": ["ram_pct", "ram_used_mb", "ram_available_mb", "swap_pct", "swap_used_mb"],
    "Disk": ["disk_usage_pct", "disk_read_mb_s", "disk_write_mb_s", "disk_latency_ms"],
    "Network": ["net_sent_mb_s", "net_recv_mb_s", "network_latency_ms"],
    "Hardware": ["temperature_c", "gpu_usage_pct", "battery_pct"],
    "System": ["process_count", "thread_count", "context_switches_per_s"],
    "Quality": ["sample_reliable", "run_complete", "missed_deadline"]
}
metric_groups = {}
for group_name, candidates in metric_candidates.items():
    metric_groups[group_name] = [column for column in candidates if column in df.columns]
    print(group_name + ":", metric_groups[group_name])

## 11. Numerical Distributions

Histograms show where values are concentrated and whether distributions are symmetric or skewed.

In [ ]:
important_metrics = [
    "cpu_pct", "ram_pct", "disk_usage_pct", "temperature_c", "gpu_usage_pct",
    "battery_pct", "process_count", "thread_count", "net_sent_mb_s", "net_recv_mb_s"
]
available_important_metrics = [column for column in important_metrics if column in df.columns]
print("Metrics to plot:", available_important_metrics)

In [ ]:
# Each metric gets its own histogram and output.
for column in available_important_metrics:
    plot_values = df[column].dropna()
    if plot_values.empty:
        print(column, "has no values to plot.")
        continue
    plt.figure(figsize=(8, 5))
    plt.hist(plot_values, bins=30)
    plt.title("Distribution of " + column)
    plt.xlabel(column)
    plt.ylabel("Number of samples")
    plt.tight_layout()
    plt.show()

## 12. Outlier Analysis

Boxplots help find values far from the usual range. Missing values are removed only from each plot.

### Goal of this visualization

A boxplot summarizes the distribution of a numerical variable.

It helps identify:
- the median,
- the spread of the data,
- the typical range of values,
- and possible outliers.

Unlike a histogram, a boxplot provides a compact statistical summary.


 Minimum      Q1        Median       Q3         Maximum
    |----------|===========|===========|-------------|
    0         10          39          61          100

In [ ]:
for column in available_important_metrics:
    plot_values = df[column].dropna()
    if plot_values.empty:
        print(column, "has no values to plot.")
        continue
    plt.figure(figsize=(8, 3))
    plt.boxplot(plot_values, vert=False)
    plt.title("Boxplot of " + column)
    plt.xlabel(column)
    plt.ylabel("Values")
    plt.tight_layout()
    plt.show()

What can you conclude from this plot?

From your boxplot, you can say:

✅ CPU usage ranges approximately from 0% to 100%.
✅ The median CPU usage is about 39%.
✅ The middle 50% of CPU measurements lie roughly between 10% and 61%.
✅ No significant outliers are visible.
✅ CPU usage shows a fairly wide spread, indicating the system experiences both low and high workloads.
🎯 One sentence to remember

A boxplot summarizes the distribution of a numerical variable by showing its minimum, first quartile (Q1), median, third quartile (Q3), maximum, and any potential outliers.


📦 Box = middle 50% of the data.
🟠 Orange line = median.
○ Circles = outliers.


### Boxplot observation

Boxplots help identify unusual observations, but they do not automatically prove that a value is incorrect. Investigate the machine, run, and collection time before deciding how to treat it.

## 13. Categorical and Quality Analysis

Study machine, run, phase, status, reliability, deadline, and battery categories.

In [ ]:
category_analysis = ["machine_id", "run_id", "phase", "status", "sample_reliable", "run_complete", "missed_deadline", "battery_plugged"]
for column in category_analysis:
    if column not in df.columns:
        print(column, "is not available; skipped.")
        continue
    print("\n", column, "counts:")
    display(df[column].value_counts(dropna=False).to_frame("count"))
    display((df[column].value_counts(normalize=True, dropna=False) * 100).round(2).to_frame("percentage"))
    if df[column].nunique(dropna=False) <= 20:
        counts = df[column].astype(str).value_counts()
        plt.figure(figsize=(9, 4))
        plt.bar(counts.index, counts.values)
        plt.title("Sample Counts by " + column)
        plt.xlabel(column)
        plt.ylabel("Number of samples")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("Chart skipped because this column has too many categories.")

### Category observation

Check whether machines and phases have balanced sample counts. Large differences may influence later comparisons.

## 14. Reliability Analysis

Measure reliable samples, complete runs, and reliability by machine.

In [ ]:
if "sample_reliable" in df.columns:
    reliability_counts = df["sample_reliable"].value_counts(dropna=False)
    reliability_percentages = (df["sample_reliable"].value_counts(normalize=True, dropna=False) * 100).round(2)
    display(pd.DataFrame({"count": reliability_counts, "percentage": reliability_percentages}))
else:
    print("sample_reliable is not available.")

In [ ]:
if "run_complete" in df.columns:
    complete_counts = df["run_complete"].value_counts(dropna=False)
    complete_percentages = (df["run_complete"].value_counts(normalize=True, dropna=False) * 100).round(2)
    display(pd.DataFrame({"count": complete_counts, "percentage": complete_percentages}))
else:
    print("run_complete is not available.")

In [ ]:
if {"machine_id", "sample_reliable"}.issubset(df.columns):
    reliability_by_machine = (df.groupby("machine_id")["sample_reliable"].mean() * 100).round(2)
    display(reliability_by_machine.to_frame("reliable_percentage"))
    plt.figure(figsize=(9, 4))
    plt.bar(reliability_by_machine.index.astype(str), reliability_by_machine.values)
    plt.title("Reliable Samples by Machine")
    plt.xlabel("Machine ID")
    plt.ylabel("Reliable samples (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Machine reliability comparison is unavailable.")

### Reliability observation

A low reliability percentage may mean missing sensors, missed deadlines, or other collection errors. Compare machines before choosing a preprocessing rule.

## 15. Analysis by Machine

Machines may have different hardware, so compare sample counts, runs, and average metrics carefully.

In [ ]:
if "machine_id" in df.columns:
    samples_by_machine = df.groupby("machine_id").size().sort_values(ascending=False)
    display(samples_by_machine.to_frame("sample_count"))
else:
    print("machine_id is not available.")

In [ ]:
if {"machine_id", "run_id"}.issubset(df.columns):
    display(df.groupby("machine_id")["run_id"].nunique().to_frame("run_count"))
else:
    print("Run counts by machine are unavailable.")

In [ ]:
machine_average_columns = [column for column in ["cpu_pct", "ram_pct", "temperature_c"] if column in df.columns]
if "machine_id" in df.columns and machine_average_columns:
    machine_averages = df.groupby("machine_id")[machine_average_columns].mean().round(2)
    display(machine_averages)
else:
    print("Machine averages are unavailable.")

In [ ]:
if "machine_id" in df.columns:
    for column in ["cpu_pct", "ram_pct", "temperature_c"]:
        if column not in df.columns or df[column].dropna().empty:
            print(column, "comparison is unavailable.")
            continue
        values = df.groupby("machine_id")[column].mean().dropna()
        plt.figure(figsize=(9, 4))
        plt.bar(values.index.astype(str), values.values)
        plt.title("Average " + column + " by Machine")
        plt.xlabel("Machine ID")
        plt.ylabel("Average " + column)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()

### Machine observation

Differences may be caused by workload, hardware, operating system, or collection conditions. They are not automatically data errors.

## 16. Analysis by Run

Compare run sizes, durations, and average CPU and RAM usage.

In [ ]:
if "run_id" in df.columns:
    samples_by_run = df.groupby("run_id").size().sort_values(ascending=False)
    print("Ten largest runs:")
    display(samples_by_run.head(10).to_frame("sample_count"))
    print("Ten smallest runs:")
    display(samples_by_run.sort_values().head(10).to_frame("sample_count"))
else:
    print("run_id is not available.")

In [ ]:
if {"run_id", "timestamp"}.issubset(df.columns):
    run_times = df.groupby("run_id")["timestamp"].agg(["min", "max"])
    run_times["duration_minutes"] = (run_times["max"] - run_times["min"]).dt.total_seconds() / 60
    print("Shortest runs:")
    display(run_times.sort_values("duration_minutes").head(10))
    print("Longest runs:")
    display(run_times.sort_values("duration_minutes", ascending=False).head(10))
else:
    print("Run duration analysis is unavailable.")

In [ ]:
run_metric_columns = [column for column in ["cpu_pct", "ram_pct"] if column in df.columns]
if "run_id" in df.columns and run_metric_columns:
    run_averages = df.groupby("run_id")[run_metric_columns].mean().round(2)
    display(run_averages.head(10))
else:
    print("Run averages are unavailable.")

### Run observation

Very short or long runs should be reviewed. Run duration can affect how representative its averages are.

## 17. Correlation Analysis

Correlation measures how two numerical variables move together. It does not prove that one variable causes another.

In [ ]:
identifier_like_columns = ["id", "legacy_id"]
correlation_columns = [column for column in numeric_columns if column not in identifier_like_columns]
if correlation_columns:
    correlation_matrix = df[correlation_columns].corr()
    display(correlation_matrix.round(2))
else:
    correlation_matrix = pd.DataFrame()
    print("No numerical columns are available for correlation.")

In [ ]:
if not correlation_matrix.empty:
    size = max(8, len(correlation_matrix) * 0.55)
    plt.figure(figsize=(size, size))
    image = plt.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(image, label="Correlation")
    plt.xticks(range(len(correlation_matrix)), correlation_matrix.columns, rotation=90)
    plt.yticks(range(len(correlation_matrix)), correlation_matrix.columns)
    plt.title("Correlation Heatmap of Numerical Metrics")
    plt.xlabel("Metric")
    plt.ylabel("Metric")
    plt.tight_layout()
    plt.show()
else:
    print("Correlation heatmap skipped.")

In [ ]:
if not correlation_matrix.empty:
    correlation_pairs = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)).stack()
    strongest_positive = correlation_pairs.sort_values(ascending=False).head(10)
    strongest_negative = correlation_pairs.sort_values().head(10)
    print("Strongest positive correlations:")
    display(strongest_positive.to_frame("correlation"))
    print("Strongest negative correlations:")
    display(strongest_negative.to_frame("correlation"))
else:
    print("Strongest correlation tables are unavailable.")

### Correlation observation

Focus on strong positive and negative relationships. Remember that correlation does not show causation.

## 18. Time-Based Analysis

Use a temporary sorted copy so the original dataframe order is not changed. Minute averages keep long series readable.

In [ ]:
if "timestamp" in df.columns and df["timestamp"].notna().any():
    time_df = df.dropna(subset=["timestamp"]).sort_values("timestamp").copy()
    print("First timestamp:", time_df["timestamp"].min())
    print("Last timestamp:", time_df["timestamp"].max())
    print("Total collection period:", time_df["timestamp"].max() - time_df["timestamp"].min())
    samples_per_minute = time_df.set_index("timestamp").resample("1min").size()
    display(samples_per_minute.head().to_frame("sample_count"))
else:
    time_df = pd.DataFrame()
    print("Valid timestamps are unavailable.")

In [ ]:
if not time_df.empty:
    samples_per_minute.plot(figsize=(10, 4))
    plt.title("Number of Samples Over Time")
    plt.xlabel("Time")
    plt.ylabel("Samples per minute")
    plt.tight_layout()
    plt.show()

In [ ]:
if not time_df.empty:
    for column in ["cpu_pct", "ram_pct", "temperature_c", "net_sent_mb_s", "net_recv_mb_s"]:
        if column not in time_df.columns or time_df[column].dropna().empty:
            print(column, "time chart is unavailable.")
            continue
        minute_values = time_df.set_index("timestamp")[column].resample("1min").mean()
        plt.figure(figsize=(10, 4))
        plt.plot(minute_values.index, minute_values.values)
        plt.title(column + " Over Time (Minute Average)")
        plt.xlabel("Time")
        plt.ylabel(column)
        plt.tight_layout()
        plt.show()

### Time observation

Look for peaks, drops, long trends, and gaps. Compare them with run phases and machines before making a preprocessing decision.

## 19. Sensor-Error Analysis

Check whether sensor error text is empty and compare it with reliability.

In [ ]:
if "sensor_errors_json" in df.columns:
    error_text = df["sensor_errors_json"].fillna("").astype(str).str.strip()
    has_sensor_error = ~error_text.isin(["", "{}", "null", "None", "nan"])
    print("Rows without sensor errors:", (~has_sensor_error).sum())
    print("Rows with sensor errors:", has_sensor_error.sum())
    print("Examples of sensor errors:")
    display(df.loc[has_sensor_error, "sensor_errors_json"].drop_duplicates().head(10))
else:
    has_sensor_error = pd.Series(False, index=df.index)
    print("sensor_errors_json is not available.")

In [ ]:
if "sensor_errors_json" in df.columns and "sample_reliable" in df.columns:
    reliability_and_errors = pd.crosstab(has_sensor_error, df["sample_reliable"], margins=True)
    reliability_and_errors.index.name = "has_sensor_error"
    display(reliability_and_errors)
else:
    print("Sensor-error and reliability comparison is unavailable.")

### Sensor-error observation

Frequent errors for one sensor or machine may explain missing values and lower reliability.

## 20. Important Validation Checks

These checks find suspicious values. They do not prove that the values are errors, and no rows are deleted.

In [ ]:
range_columns = ["cpu_pct", "ram_pct", "disk_usage_pct", "battery_pct"]
for column in range_columns:
    if column in df.columns:
        suspicious = df[(df[column] < 0) | (df[column] > 100)]
        print(column, "outside 0 to 100:", len(suspicious))
        if not suspicious.empty:
            display(suspicious[[column]].head(10))
    else:
        print(column, "is not available; skipped.")

In [ ]:
non_negative_columns = [
    "net_sent_mb_s", "net_recv_mb_s", "disk_read_mb_s", "disk_write_mb_s",
    "process_count", "thread_count"
]
for column in non_negative_columns:
    if column in df.columns:
        suspicious = df[df[column] < 0]
        print(column, "negative values:", len(suspicious))
        if not suspicious.empty:
            display(suspicious[[column]].head(10))
    else:
        print(column, "is not available; skipped.")

### Validation observation

Investigate suspicious rows using machine, run, timestamp, and phase. Some unusual values may still be valid.

## Key Findings and Observations

Complete this template after running and interpreting the notebook:

- **Dataset size:**
- **Number of machines:**
- **Number of runs:**
- **Collection period:**
- **Main missing-value problems:**
- **Percentage of reliable samples:**
- **Percentage of complete runs:**
- **Main numerical distributions:**
- **Main outliers:**
- **Strongest correlations:**
- **Differences between machines:**
- **Important time-based patterns:**
- **Suspicious or invalid values:**
- **Decisions required for preprocessing:**

## Final Conclusion

EDA helped us understand the structure and quality of the system metrics. No preprocessing decision should be made before interpreting the results.

The next step is preprocessing, including missing-value treatment, feature selection, encoding, scaling, and possible outlier treatment.

The original cleaned dataset was not modified.